# Baseline — ce qu'un modèle de lettres tout fait écrit de nos prises

Aucun entraînement. C'est le chiffre contre lequel l'affinage se jugera : ce qu'un
modèle nourri de livres lus rend des hésitations d'un apprenant.

Il sort du majuscule sans ponctuation, comme tout CTC — c'est la forme, pas un défaut.

Réglages Kaggle : accélérateur **GPU T4**, Internet **on**, **aucune persistance**.
Dataset à attacher : `gilleslandrin/saylune-hesitation-takes`.

Ce qui rentre à la maison : `baseline.json`, quelques kilo-octets.

In [ ]:
MODEL = "facebook/wav2vec2-large-960h-lv60-self"

In [ ]:
import json, pathlib

import soundfile
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(MODEL)
model = Wav2Vec2ForCTC.from_pretrained(MODEL).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"{MODEL}: {sum(p.numel() for p in model.parameters()):,} parameters on {device}")

In [ ]:
# Found rather than guessed: Kaggle nests a mounted dataset at a depth
# that is not the same from one attachment to the next.
root = next(pathlib.Path("/kaggle/input").rglob("references.json")).parent
references = json.loads((root / "references.json").read_text(encoding="utf-8"))
asked = json.loads((root / "answers" / "asked.json").read_text(encoding="utf-8"))
takes = sorted(root.rglob("*.wav"))
print(len(takes), "takes under", root)

In [ ]:
rows = []
for wav in takes:
    which, slug = wav.parent.name, wav.stem
    audio, rate = soundfile.read(wav)
    if rate != 16000:
        raise SystemExit(f"{wav}: {rate} Hz, expected 16000")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    heard = processor(audio, sampling_rate=rate, return_tensors="pt")
    with torch.no_grad():
        logits = model(heard.input_values.to(device),
                       attention_mask=heard.attention_mask.to(device)).logits
    text = processor.batch_decode(logits.argmax(dim=-1))[0]
    row = {"set": which, "slug": slug, "verbatim": text}
    if which == "stumbles":
        row["reference"] = references[slug]["text"]
        row["kind"] = references[slug]["kind"]
    else:
        row["asked"] = asked.get(slug)
    rows.append(row)
    print(f"\n[{which}/{slug}]")
    if "reference" in row:
        print(f"  attendu : {row['reference']}")
    print(f"  rendu   : {text}")

In [ ]:
out = pathlib.Path("/kaggle/working/baseline.json")
out.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(out, out.stat().st_size, "bytes")